## Deploying a Workflow Agent to Vertex AI Agent Engine

This notebook takes the verified answer pipeline from Challenge 4 and
deploys it to **Vertex AI Agent Engine** -- Google Cloud's managed
runtime for production agents.

Steps:
1. Install dependencies and configure Vertex AI
2. Define the agent (reused from Challenge 4)
3. Test locally with `AdkApp`
4. Deploy to Agent Engine with `agent_engines.create()`
5. Test the deployed agent remotely
6. Clean up

## 1. Setup & Dependencies

In [ ]:
!pip install google-cloud-aiplatform[agent_engines,adk] -q
!pip install google-genai -q

## 2. Configuration

Initialize Vertex AI with project, location, and a staging bucket
for Agent Engine deployment artifacts.

In [ ]:
import os
from typing import Optional

import google.auth
import vertexai
from google.genai import types
from google import genai

from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent, LlmAgent
from google.adk.tools.tool_context import ToolContext
from vertexai import agent_engines
from vertexai.agent_engines import AdkApp

project = google.auth.default()[1]
LOCATION = "us-central1"
GEMINI_MODEL = "gemini-2.0-flash"
STAGING_BUCKET = f"gs://{project}-agent-engine-staging"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = project
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(
    project=project,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"Project:        {project}")
print(f"Location:       {LOCATION}")
print(f"Staging bucket: {STAGING_BUCKET}")
print(f"Model:          {GEMINI_MODEL}")

### Create the staging bucket (if it doesn't exist)

In [ ]:
!gsutil ls $STAGING_BUCKET 2>/dev/null || gsutil mb -l $LOCATION $STAGING_BUCKET
print(f"Staging bucket ready: {STAGING_BUCKET}")

## 3. Tools

Same tools from Challenge 4 with one key change: `search_the_web`
creates its own GenAI client inside the function so it is fully
self-contained and serializable for Agent Engine deployment.

In [ ]:
def search_the_web(query: str) -> str:
    """Search the web for information using Google Search.

    Use this tool to find up-to-date information about any topic.

    Args:
        query: The search query to look up.

    Returns:
        A text summary of the search results.
    """
    from google import genai as _genai
    from google.genai import types as _types
    import google.auth as _auth

    _project = _auth.default()[1]
    client = _genai.Client(vertexai=True, project=_project, location="us-central1")
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=query,
        config=_types.GenerateContentConfig(
            tools=[_types.Tool(google_search=_types.GoogleSearch())],
        ),
    )
    return response.text


def set_state_value(
    tool_context: ToolContext, field: str, value: str
) -> dict:
    """Set a single value in session state, overwriting any existing value.

    Use this to store a draft answer, critique, or any single value
    that downstream agents need to read via template substitution.

    Args:
        field: The state key to set (e.g., 'current_answer').
        value: The text value to store.

    Returns:
        A status dict confirming success.
    """
    tool_context.state[field] = value
    return {"status": "success", "field": field}


def append_to_state(
    tool_context: ToolContext, field: str, value: str
) -> dict:
    """Append a value to a list stored in session state.

    Use this to keep a running log of observations, critiques,
    or refinements across iterations.

    Args:
        field: The state key to append to (e.g., 'critique_history').
        value: The text value to append.

    Returns:
        A status dict confirming success.
    """
    existing = tool_context.state.get(field, [])
    tool_context.state[field] = existing + [value]
    return {"status": "success", "entries": len(existing) + 1}

## 4. Agent Definitions

The same verified answer pipeline from Challenge 4:

```
root_agent (Agent -- LLM router)
|-- greeter_agent (LlmAgent)
|-- answer_pipeline (SequentialAgent)
    |-- research_phase (ParallelAgent)
    |   |-- search_agent    -> state["search_results"]
    |   |-- context_agent   -> state["background_context"]
    |-- synthesize_agent    -> state["current_answer"]
    |-- refinement_loop (LoopAgent, max_iterations=2)
        |-- critique_agent  -> state["critique"]
        |-- refine_agent    -> state["current_answer"] (overwritten)
```

Note: Callbacks are omitted for Agent Engine deployment since logging
happens server-side.

In [ ]:
greeter_agent = LlmAgent(
    name="greeter_agent",
    model=GEMINI_MODEL,
    description=(
        "Handles greetings and casual conversation. Use this agent "
        "when the user says hello, hi, thanks, goodbye, or engages "
        "in small talk rather than asking a factual question."
    ),
    instruction="""
You are a friendly, helpful greeter. Respond warmly to the user's
greeting or casual message. Keep it brief and cheerful. If they
seem to have a question, encourage them to ask it.
""",
)

search_agent = LlmAgent(
    name="search_agent",
    model=GEMINI_MODEL,
    description="Searches for the direct answer to a user's question.",
    instruction="""
You are a research agent. Use the search_the_web tool to find a
direct, factual answer to the user's question.

Return ONLY the key facts and data points you found. Include
specific numbers, dates, names, and sources where possible.
Do not add opinions or commentary -- just the raw findings.
""",
    tools=[search_the_web],
    output_key="search_results",
)

context_agent = LlmAgent(
    name="context_agent",
    model=GEMINI_MODEL,
    description="Finds background context and related information.",
    instruction="""
You are a background research agent. Use the search_the_web tool
to find additional context, history, or related information that
would enrich the answer to the user's question.

Focus on:
- Historical context or background
- Related facts that add depth
- Why this topic matters or is interesting

Return ONLY the supplementary facts. Do not repeat the direct
answer -- focus on what makes the answer richer.
""",
    tools=[search_the_web],
    output_key="background_context",
)

research_phase = ParallelAgent(
    name="research_phase",
    sub_agents=[search_agent, context_agent],
    description="Runs search and context agents in parallel.",
)

synthesize_agent = LlmAgent(
    name="synthesize_agent",
    model=GEMINI_MODEL,
    description="Combines research into a draft answer.",
    instruction="""
You are a synthesis agent. Combine the following research into a
clear, well-structured draft answer.

## Direct Search Results
{search_results}

## Background Context
{background_context}

Write a comprehensive answer that:
1. Leads with the direct answer to the question
2. Adds relevant context and background
3. Cites sources where available
4. Is well-organized and easy to read

IMPORTANT: After writing your answer, you MUST use the set_state_value
tool to store your complete answer with field='current_answer'.
Also use the append_to_state tool to log this draft to the
'answer_history' field for tracking progress across refinements.
""",
    tools=[set_state_value, append_to_state],
    output_key="current_answer",
)

critique_agent = LlmAgent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Evaluates the current answer and suggests improvements.",
    instruction="""
You are a critical reviewer. Evaluate the following answer and
provide specific, actionable suggestions for improvement.

## Current Answer
{current_answer}

Evaluate against these criteria:
- **Accuracy**: Are the facts correct and well-sourced?
- **Completeness**: Does it fully answer the question?
- **Clarity**: Is it well-organized and easy to understand?
- **Conciseness**: Is there unnecessary filler or repetition?

Provide your critique as a numbered list of specific improvements.
Be constructive -- explain what to change AND why.

IMPORTANT: After writing your critique, you MUST use the set_state_value
tool to store your complete critique with field='critique'.
Also use the append_to_state tool to log your critique to the
'critique_history' field.
""",
    tools=[set_state_value, append_to_state],
    output_key="critique",
)

refine_agent = LlmAgent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Rewrites the answer based on critique feedback.",
    instruction="""
You are a refinement agent. Rewrite the answer based on the
critique feedback provided.

## Current Answer
{current_answer}

## Critique & Suggestions
{critique}

Rewrite the answer addressing ALL of the critique points.
Maintain the good parts while improving the weak areas.
The result should be a polished, final-quality response.

IMPORTANT: After writing your refined answer, you MUST use the
set_state_value tool to store your complete refined answer with
field='current_answer'. Also use the append_to_state tool to log
this refined version to the 'answer_history' field.
""",
    tools=[set_state_value, append_to_state],
    output_key="current_answer",
)

refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critique_agent, refine_agent],
    max_iterations=2,
    description="Iteratively critiques and refines the answer.",
)

answer_pipeline = SequentialAgent(
    name="answer_pipeline",
    description=(
        "Answers factual questions using a verified research pipeline. "
        "Use this for any question that requires looking up information, "
        "facts, history, science, current events, how-to questions, etc."
    ),
    sub_agents=[research_phase, synthesize_agent, refinement_loop],
)

root_agent = Agent(
    name="root_agent",
    model=GEMINI_MODEL,
    description="The main coordinating agent that routes user requests.",
    instruction="""
You are the main coordinating agent. Your job is to understand what
the user needs and delegate to the right specialist:

- **greeter_agent**: For greetings, casual conversation, thanks,
  or goodbyes.
- **answer_pipeline**: For ANY factual question, lookup, or request
  that requires searching for information.

Always delegate to a sub-agent. Do not try to answer questions
yourself. If the user's intent is unclear, ask a brief clarifying
question.
""",
    sub_agents=[greeter_agent, answer_pipeline],
)

print("Agent hierarchy defined successfully.")

## 5. Local Testing with AdkApp

Wrap the agent in `AdkApp` and test locally before deploying.
This uses in-memory sessions and validates that the agent works
end-to-end.

In [ ]:
app = AdkApp(agent=root_agent)
print("AdkApp created successfully.")

### 5a. Local Test -- Greeting

In [ ]:
session = app.create_session(user_id="test_user")
print(f"Session ID: {session.id}\n")

for event in app.stream_query(
    user_id="test_user",
    session_id=session.id,
    message="Hello! How are you today?",
):
    if hasattr(event, 'content') and event.content and event.content.parts:
        for part in event.content.parts:
            if hasattr(part, 'text') and part.text:
                print(f"  [{getattr(event, 'author', 'agent')}] {part.text[:300]}")

### 5b. Local Test -- Factual Question (Full Pipeline)

In [ ]:
session2 = app.create_session(user_id="test_user")
print(f"Session ID: {session2.id}\n")

for event in app.stream_query(
    user_id="test_user",
    session_id=session2.id,
    message="Who won the most recent Super Bowl and what were the key highlights?",
):
    if hasattr(event, 'content') and event.content and event.content.parts:
        for part in event.content.parts:
            if hasattr(part, 'text') and part.text:
                print(f"  [{getattr(event, 'author', 'agent')}] {part.text[:200]}{'...' if len(part.text) > 200 else ''}")
            elif hasattr(part, 'function_call') and part.function_call:
                print(f"  [{getattr(event, 'author', 'agent')}] -> {part.function_call.name}()")

## 6. Deploy to Agent Engine

Deploy the agent to Vertex AI Agent Engine using `agent_engines.create()`.
This packages the agent, uploads it to the staging bucket, and creates
a managed endpoint. This may take several minutes.

In [ ]:
REQUIREMENTS = [
    "google-cloud-aiplatform[agent_engines,adk]",
    "google-genai",
    "cloudpickle",
]

remote_agent = agent_engines.create(
    agent_engine=app,
    requirements=REQUIREMENTS,
    display_name="verified-answer-pipeline",
)

print(f"\nDeployed successfully!")
print(f"Resource name: {remote_agent.resource_name}")

## 7. Test the Deployed Agent

Now test the agent running on Agent Engine. This uses cloud-managed
sessions instead of in-memory sessions.

### 7a. Remote Test -- Greeting

In [ ]:
remote_session = remote_agent.create_session(user_id="remote_test_user")
print(f"Remote session: {remote_session}\n")

for event in remote_agent.stream_query(
    user_id="remote_test_user",
    session_id=remote_session["id"],
    message="Hi there! Good morning!",
):
    print(event)

### 7b. Remote Test -- Factual Question

In [ ]:
remote_session2 = remote_agent.create_session(user_id="remote_test_user")
print(f"Remote session: {remote_session2}\n")

for event in remote_agent.stream_query(
    user_id="remote_test_user",
    session_id=remote_session2["id"],
    message="What is the James Webb Space Telescope and what has it discovered?",
):
    print(event)

## 8. Clean Up

Delete the deployed agent to avoid ongoing charges.
Uncomment and run the cell below when you are done testing.

In [ ]:
# Uncomment to delete the deployed agent:
# remote_agent.delete()
# print("Agent Engine resource deleted.")